# Lesson 1: From Tutorial 1's Tiny GPT to GPT-2

**Question:** What changed when the tiny character-level language model became GPT-2 (124M)?

We will compare the vocabulary and model dimensions, read the shapes used by attention, and estimate the parameter count without allocating weights or downloading a checkpoint. This makes later implementation choices easier to understand.

First tutorial link: [Lesson 14, train the Tiny Shakespeare decoder](../../../gpt-1/python/lesson_14_train_decoder_language_model.ipynb).
GPT-2 source: [GPTConfig and GPT](../../build-nanogpt/train_gpt2.py).

The first tutorial's Tiny Shakespeare run uses 65 characters, a context of 16, two 16-channel blocks, and two attention heads. GPT-2 uses a byte-level BPE tokenizer, a context of 1,024 tokens, twelve 768-channel blocks, and twelve heads. The tokenizer can encode text more compactly than one character per token; we will inspect how it assigns IDs in the tokenizer lesson.


In [ ]:
from dataclasses import replace
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd() / "gpt-2", *Path.cwd().parents]
tutorial_root = next(
    (candidate for candidate in candidates
     if (candidate / "experiments/configs/tiny-playground.json").is_file()),
    None,
)
if tutorial_root is None:
    raise FileNotFoundError("Open the repository, gpt-2 folder, or its notebook folder and restart the kernel.")

from gpt2_lab.architecture import gpt2_parameter_breakdown
from gpt2_lab.config import ModelConfig

print(f"Python: {sys.executable}")
print(f"GPT-2 project: {tutorial_root}")


## 1. Compare the configurations

V is vocabulary size, T is maximum context length, L is block count, H is attention head count, and C is the residual stream width (n_embd). The head size is C / H.

B, the batch size, is chosen for a training run. It changes how many examples run together, not how many learned model parameters exist. None of these configuration objects allocates a model.


In [ ]:
tutorial1 = ModelConfig.tutorial1_tiny()
gpt2 = ModelConfig.gpt2_124m()

print(f"{'model':<20} {'V':>7} {'T':>6} {'L':>4} {'H':>4} {'C':>5} {'head':>6}")
for name, config in (("Tutorial 1 tiny model", tutorial1), ("GPT-2 124M", gpt2)):
    print(f"{name:<20} {config.vocab_size:>7,} {config.context_length:>6,} "
          f"{config.n_layer:>4} {config.n_head:>4} {config.n_embd:>5,} {config.head_size:>6}")


The first tutorial maps each distinct character in Tiny Shakespeare to an ID. GPT-2 maps byte sequences to BPE token IDs: the standard vocabulary has 50,257 entries (256 byte tokens, 50,000 merge tokens, and one end-of-text token).

The model config uses V=50,257. In the training script, Karpathy pads the output vocabulary to 50,304 for faster hardware alignment. The extra 47 model rows do not add tokenizer IDs or data tokens.


## 2. Follow one token through attention

Input IDs have shape (B, T). Token and position lookups turn each ID into a vector, forming hidden states of shape (B, T, C). GPT-2's fused QKV projection produces three 768-wide vectors for each position. The width is split across heads and arranged as (B, H, T, C/H), which here is (B, 12, T, 64). The causal attention score matrix has shape (B, H, T, T).

After twelve Transformer blocks, the output head maps the final width to vocabulary logits of shape (B, T, V). This shape comparison uses the canonical tokenizer vocabulary V=50,257; the training script's padded output has 50,304 logits.


In [ ]:
print(f"input IDs                    (B, T)        -> (B, T)")
print(f"token and position vectors   (B, T, C)     -> (B, T, {gpt2.n_embd})")
print(f"queries, keys, values         (B, H, T, C/H) -> (B, {gpt2.n_head}, T, {gpt2.head_size}) each")
print(f"causal attention scores       (B, H, T, T)   -> (B, {gpt2.n_head}, T, T)")
print(f"vocabulary logits             (B, T, V)      -> (B, T, {gpt2.vocab_size:,})")


## 3. Estimate parameters from the shapes

Let C be residual width, V vocabulary, T context, and L block count. The shared GPT-2 block has:

- Fused query/key/value projection: 3C² + 3C weights and biases.
- Attention output projection: C² + C.
- MLP expansion and projection: (4C² + 4C) + (4C² + C).
- Two LayerNorms, each with scale and shift: 4C.

Together, a block uses 12C² + 13C learned parameters. Add token embeddings VC, position embeddings TC, and the final LayerNorm 2C.

GPT-2 ties the output projection's weight matrix to the token embedding table. The output head therefore adds zero additional weight parameters (and it has no bias). This saves one V x C matrix, about 38.6 million parameters in GPT-2 small. The count below uses the 50,257-token checkpoint vocabulary.


In [ ]:
breakdown = gpt2_parameter_breakdown(gpt2)
for component, count in breakdown.items():
    print(f"{component:<32} {count:>15,}")
parameter_count = sum(breakdown.values())
print(f"{'total':<32} {parameter_count:>15,}  ({parameter_count / 1_000_000:.3f} million)")
padded_gpt2 = replace(gpt2, vocab_size=50_304)
padded_count = sum(gpt2_parameter_breakdown(padded_gpt2).values())
print(f"Training config with 50,304 output rows: {padded_count:,} (+{padded_count - parameter_count:,})")


## Compare C and Java

The companions repeat the same three steps: compare the two configurations, calculate head width and tensor shapes, and estimate GPT-2's parameter count.

- C: [architecture_lesson.c](../../c/src/architecture_lesson.c) and the [shared configuration helpers](../../c/src/model_config.c).
- Java: [ArchitectureLesson.java](../../java/src/main/java/learning/karpathy/gpt2/ArchitectureLesson.java) and [ModelConfig.java](../../java/src/main/java/learning/karpathy/gpt2/ModelConfig.java).

From the repository root, run all steps or select one in either language:

```powershell
.\scripts\run-gpt2.ps1 -Language c -Action lesson
.\scripts\run-gpt2.ps1 -Language java -Action lesson -Part 2
```

VS Code also has **GPT-2: Run Lesson 1** under **Terminal → Run Task**. The Python notebook remains the entry point and explanation; C and Java place the same steps in ordinary functions and a class.


## Try changing the model

Use a copy of the baseline configuration so the original stays available for comparison. Change only one setting and predict the effect first:

1. Add 1,000 vocabulary entries. How many parameters does the tied token/output table gain?
2. Increase context length by 128 positions. Which learned table grows, and by how many parameters?
3. Keep width fixed and change the head count from 12 to 16. Does the number of head channels change? Does the parameter count change in this architecture?

A dataclasses.replace example lets us reason about a shape change without allocating model weights:


In [ ]:
larger_vocabulary = replace(gpt2, vocab_size=gpt2.vocab_size + 1_000)
longer_context = replace(gpt2, context_length=gpt2.context_length + 128)
more_heads = replace(gpt2, n_head=16)

print(f"+1,000 tokens -> +{gpt2_parameter_breakdown(larger_vocabulary)['token_embeddings'] - breakdown['token_embeddings']:,} embedding parameters")
print(f"+128 positions -> +{gpt2_parameter_breakdown(longer_context)['position_embeddings'] - breakdown['position_embeddings']:,} position parameters")
print(f"16 heads -> head size {more_heads.head_size}; total parameters "
      f"{sum(gpt2_parameter_breakdown(more_heads).values()):,}")


For a dataset or architecture idea you want to try, record it through the experiment workflow. This lesson only inspects settings and does not train anything. The next lesson will examine GPT-2 tokenization and shared token examples.
